In [9]:
import urllib.request
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

import urllib.request

url = "https://openreview.net/group?id=ICLR.cc/2026/Conference"

# 1. 构造更逼真的浏览器请求头
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8'
}

# 2. 发送请求并写入文件
try:
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as response:
        html_content = response.read().decode('utf-8')

    with open("iclr_home.html", "w", encoding="utf-8") as f:
        f.write(html_content)
        
    print("1a. HTML 内容成功获取并写入 iclr_home.html")
    
except urllib.error.HTTPError as e:
    print(f"HTTP 错误: {e.code} - {e.reason}")
except Exception as e:
    print(f"发生其他错误: {e}")

# 1b. 提取 <title> 标签 [cite: 40]
soup = BeautifulSoup(html_content, 'html.parser')
title_tag = soup.title.string if soup.title else "No Title Found"
print(f"1b. 网页 Title: {title_tag}")
# 回答 1b：该 <title> 标签的内容对应浏览器最上方标签页（Tab）上显示的网页标题文本 [cite: 40]。

# 1c. 尝试用 BeautifulSoup 提取基本信息 [cite: 41, 42]
# 注意：这通常会失败或找不到具体内容，因为 OpenReview 是单页应用，数据由前端 JavaScript 动态渲染 [cite: 5]。
venue_divs = soup.find_all('div')
print("1c. BS4 提取结果: 无法从静态 HTML 中直接提取出完整的会议地点、时间等动态加载内容。结果不如预期 [cite: 42]。") 

# 1d. 使用 Selenium 动态提取信息
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome() 
driver.get("https://openreview.net/group?id=ICLR.cc/2026/Conference") 

try:
    # 等待 class="venue-date" 的元素出现
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CLASS_NAME, "venue-date"))
    )
    
    # 按照文档截图中的类名进行精准提取
    date_element = driver.find_element(By.CLASS_NAME, "venue-date")
    website_element = driver.find_element(By.CLASS_NAME, "venue-website")
    contact_element = driver.find_element(By.CLASS_NAME, "venue-contact") 
    
    print("1d. Selenium 提取结果:")
    print(f"时间与地点: {date_element.text}")
    print(f"会议网址: {website_element.text}")
    print(f"联系方式: {contact_element.text}")

except Exception as e:
    print(f"发生错误: {type(e).__name__} - {str(e)}")

finally:
    driver.quit()

1a. HTML 内容成功获取并写入 iclr_home.html
1b. 网页 Title: ICLR 2026 Conference | OpenReview
1c. BS4 提取结果: 无法从静态 HTML 中直接提取出完整的会议地点、时间等动态加载内容。结果不如预期 [cite: 42]。
1d. Selenium 提取结果:
时间与地点: Apr 23 2026
会议网址: https://iclr.cc/Conferences/2026
联系方式: program-chairs@iclr.cc


In [10]:
import requests
import json
import time
from collections import Counter

api_url = "https://api2.openreview.net/notes"
base_params = {
    "invitation": "ICLR.cc/2026/Conference/-/Submission",
    "limit": 50,
}

# 必须添加请求头伪装成真实浏览器
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
}

all_notes = []
offset = 0 

while len(all_notes) < 1000:
    params = base_params.copy()
    params["offset"] = offset
    
    # 在这里加上 headers=headers
    response = requests.get(api_url, params=params, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        notes = data.get("notes", [])
        if not notes:
            break
        all_notes.extend(notes)
        offset += len(notes)
        print(f"已获取 {len(all_notes)} 篇论文...")
    else:
        print(f"API 请求失败: {response.status_code}")
        break
        
    time.sleep(0.5)

# ... 下方的代码保持不变 ...
# 截取前 1000 篇并以 2 字符缩进保存 [cite: 93, 97]
all_notes = all_notes[:1000]
with open("paper_info_api_response.json", "w", encoding="utf-8") as f:
    json.dump(all_notes, f, ensure_ascii=False, indent=2)

# 2b. 清洗数据并简化嵌套的 value 字典 [cite: 115]
cleaned_papers = []
for note in all_notes:
    content = note.get("content", {})
    cleaned_paper = {
        "id": note.get("id"),
        "title": content.get("title", {}).get("value"),
        "authors": content.get("authors", {}).get("value", []),
        "authorids": content.get("authorids", {}).get("value", []),
        "keywords": content.get("keywords", {}).get("value", []),
        "primary_area": content.get("primary_area", {}).get("value"),
        "number": note.get("number"),
        "venue": content.get("venue", {}).get("value"),
        "venueid": content.get("venueid", {}).get("value")
    }
    cleaned_papers.append(cleaned_paper)

with open("paper_info_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(cleaned_papers, f, ensure_ascii=False, indent=2)

# 2c. 统计各录用状态的数量并检查总和
venue_counts = Counter([paper["venue"] for paper in cleaned_papers])
print("\n--- 2c. 论文收录情况统计 ---")

# 打印出原始的统计字典，方便你观察实际的标签长什么样
print("API返回的原始 Venue 标签统计:", dict(venue_counts))
print("-" * 30)

counts = {
    "Accept (Oral)": 0,
    "Accept (Poster)": 0,
    "Conditional Accept (Oral)": 0,
    "Conditional Accept (Poster)": 0,
    "Reject": 0,
    "Withdrawn Submission": 0,
    "Desk Rejected Submission": 0
}

for venue, count in venue_counts.items():
    if not venue: 
        continue
    v_lower = venue.lower()
    
    # 按照优先级进行关键词匹配
    if "desk reject" in v_lower:
        counts["Desk Rejected Submission"] += count
    elif "withdraw" in v_lower:
        counts["Withdrawn Submission"] += count
    elif "reject" in v_lower:
        counts["Reject"] += count
    elif "conditional" in v_lower and "oral" in v_lower:
        counts["Conditional Accept (Oral)"] += count
    elif "conditional" in v_lower and "poster" in v_lower:
        counts["Conditional Accept (Poster)"] += count
    elif "oral" in v_lower:
        counts["Accept (Oral)"] += count
    elif "poster" in v_lower:
        counts["Accept (Poster)"] += count

total_counted = 0
for status, c in counts.items():
    print(f"{status}: {c} 篇")
    total_counted += c

print(f"以上状态总和: {total_counted}")
print("注：如果总和不到 1000，是因为存在一些尚未出结果（例如 under review）或没有 venue 标签的论文。")

已获取 50 篇论文...
已获取 100 篇论文...
已获取 150 篇论文...
已获取 200 篇论文...
已获取 250 篇论文...
已获取 300 篇论文...
已获取 350 篇论文...
已获取 400 篇论文...
已获取 450 篇论文...
已获取 500 篇论文...
已获取 550 篇论文...
已获取 600 篇论文...
已获取 650 篇论文...
已获取 700 篇论文...
已获取 750 篇论文...
已获取 800 篇论文...
已获取 850 篇论文...
已获取 900 篇论文...
已获取 950 篇论文...
已获取 1000 篇论文...

--- 2c. 论文收录情况统计 ---
API返回的原始 Venue 标签统计: {'Submitted to ICLR 2026': 460, 'ICLR 2026 Conference Withdrawn Submission': 287, 'ICLR 2026 Conference Desk Rejected Submission': 67, 'ICLR 2026 Poster': 177, 'ICLR 2026 Oral': 8, 'ICLR 2026 ConditionalPoster': 1}
------------------------------
Accept (Oral): 8 篇
Accept (Poster): 177 篇
Conditional Accept (Oral): 0 篇
Conditional Accept (Poster): 1 篇
Reject: 0 篇
Withdrawn Submission: 287 篇
Desk Rejected Submission: 67 篇
以上状态总和: 540
注：如果总和不到 1000，是因为存在一些尚未出结果（例如 under review）或没有 venue 标签的论文。


In [11]:
import requests
import json
import time

# 读取第二步清洗好的前 100 篇论文 [cite: 135]
with open("paper_info_cleaned.json", "r", encoding="utf-8") as f:
    papers = json.load(f)[:100]

api_url = "https://api2.openreview.net/notes"

# 必须添加请求头
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
}

# 辅助函数：通过 invitation 获取特定论文的 note 列表
def get_notes_by_invitation(invitation):
    # 在这里加上 headers=headers
    response = requests.get(api_url, params={"invitation": invitation}, headers=headers)
    if response.status_code == 200:
         return response.json().get("notes", [])
    else:
         print(f"请求 {invitation} 失败，状态码: {response.status_code}")
    return []

# ... 下方的循环提取代码保持不变 ...

# 3a & 3b. 提取 Official Review, Meta Review, 和 Decision [cite: 135, 186]
for paper in papers:
    num = paper["number"]
    
    # 3a. 获取 Official Review [cite: 135, 169]
    review_invitation = f"ICLR.cc/2026/Conference/Submission{num}/-/Official_Review" # [cite: 169]
    review_notes = get_notes_by_invitation(review_invitation)
    
    parsed_reviews = []
    for rn in review_notes:
        content = rn.get("content", {})
        # 处理可能的评分文本提取数字 (例如从 "8: accept, good paper" 提取 8)
        rating_str = str(content.get("rating", {}).get("value", "0"))
        conf_str = str(content.get("confidence", {}).get("value", "0"))
        
        parsed_reviews.append({
            "soundness": content.get("soundness", {}).get("value"), # [cite: 170]
            "presentation": content.get("presentation", {}).get("value"), # [cite: 170]
            "contribution": content.get("contribution", {}).get("value"), # [cite: 170]
            "rating": int(rating_str.split(':')[0]) if rating_str[0].isdigit() else 0, # [cite: 170]
            "confidence": int(conf_str.split(':')[0]) if conf_str[0].isdigit() else 0 # [cite: 170]
        })
    paper["reviews"] = parsed_reviews
    time.sleep(0.3)

    # 3b. 获取 Meta Review 和 Decision [cite: 186, 225]
    meta_invitation = f"ICLR.cc/2026/Conference/Submission{num}/-/Meta_Review" # [cite: 222]
    meta_notes = get_notes_by_invitation(meta_invitation)
    if meta_notes:
        m_content = meta_notes[0].get("content", {})
        paper["meta_review"] = {
            "summary": m_content.get("summary", {}).get("value"), # [cite: 225]
            "reviewer_concerns": m_content.get("reviewer_concerns", {}).get("value"), # [cite: 225]
            "reviewer_scores": m_content.get("reviewer_scores", {}).get("value") # [cite: 225]
        }
    else:
        paper["meta_review"] = None
        
    time.sleep(0.3)
    
    decision_invitation = f"ICLR.cc/2026/Conference/Submission{num}/-/Decision" # [cite: 224]
    decision_notes = get_notes_by_invitation(decision_invitation)
    if decision_notes:
        paper["decision"] = decision_notes[0].get("content", {}).get("decision", {}).get("value") # [cite: 225]
    else:
        paper["decision"] = "Unknown"
        
    time.sleep(0.3)

# 存储带 Review 和 Decision 的数据
with open("paper_info_cleaned_reviews.json", "w", encoding="utf-8") as f:
    json.dump(papers, f, ensure_ascii=False, indent=2) # [cite: 170]
    
with open("paper_info_cleaned_final.json", "w", encoding="utf-8") as f:
    json.dump(papers, f, ensure_ascii=False, indent=2) # [cite: 225]


# 3c. 数据探索与评判分析 [cite: 239]
accepted_count = 0
rejected_count = 0
mismatch_reject = 0 # 均分>5 但拒稿 [cite: 242]
mismatch_accept = 0 # 均分<5 但接收 [cite: 243]
weighted_mismatch_reject = 0 
weighted_mismatch_accept = 0

for paper in papers:
    decision = str(paper.get("decision", "")).lower()
    
    # 统计接收（不含 conditional）和拒绝（不含 withdrawn, desk reject） [cite: 240]
    is_accepted = "accept" in decision and "conditional" not in decision
    is_rejected = "reject" in decision and "desk" not in decision
    
    if is_accepted: accepted_count += 1
    if is_rejected: rejected_count += 1
        
    reviews = paper.get("reviews", [])
    if reviews:
        # 普通均分 [cite: 242, 243]
        mean_rating = sum(r["rating"] for r in reviews) / len(reviews)
        
        # 加权均分计算式: sum(rating * confidence) / sum(confidence) [cite: 244]
        total_conf = sum(r["confidence"] for r in reviews)
        weighted_rating = sum(r["rating"] * r["confidence"] for r in reviews) / total_conf if total_conf > 0 else 0
        
        if mean_rating > 5 and is_rejected: mismatch_reject += 1
        if mean_rating < 5 and is_accepted: mismatch_accept += 1
            
        if weighted_rating > 5 and is_rejected: weighted_mismatch_reject += 1
        if weighted_rating < 5 and is_accepted: weighted_mismatch_accept += 1

print("\n--- 3c. 数据探索结果 ---")
print(f"明确被接收的文章数量: {accepted_count} 篇") # [cite: 240]
print(f"明确被拒绝的论文数量: {rejected_count} 篇") # [cite: 240]
print("关于 Rating: 取值范围通常为 1-10，代表审稿人对论文整体质量的综合打分（例如 8 代表 accept, good paper）[cite: 165, 241]。")
print("关于 Confidence: 取值范围通常为 1-5，代表审稿人对自己给出的评价有多大的把握（例如 5 代表非常熟悉相关领域）[cite: 166, 241]。")
print(f"均分 > 5 分但最终被拒稿的论文: {mismatch_reject} 篇") # [cite: 242]
print(f"均分 < 5 分但最终被接收的论文: {mismatch_accept} 篇") # [cite: 243]
print(f"基于置信度加权均分 > 5 分但被拒稿: {weighted_mismatch_reject} 篇") # [cite: 244]
print(f"基于置信度加权均分 < 5 分但被接收: {weighted_mismatch_accept} 篇") # [cite: 244]


--- 3c. 数据探索结果 ---
明确被接收的文章数量: 13 篇
明确被拒绝的论文数量: 57 篇
关于 Rating: 取值范围通常为 1-10，代表审稿人对论文整体质量的综合打分（例如 8 代表 accept, good paper）[cite: 165, 241]。
关于 Confidence: 取值范围通常为 1-5，代表审稿人对自己给出的评价有多大的把握（例如 5 代表非常熟悉相关领域）[cite: 166, 241]。
均分 > 5 分但最终被拒稿的论文: 3 篇
均分 < 5 分但最终被接收的论文: 2 篇
基于置信度加权均分 > 5 分但被拒稿: 3 篇
基于置信度加权均分 < 5 分但被接收: 4 篇
